In [1]:
import os, glob, json, zipfile, logging, warnings, shutil
import datetime as dt
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_bounds
from pathlib import Path

os.chdir(os.path.expanduser("~/projects/iride_onboard-burnscar-mapper"))
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("dataset")

# ---------------- SHARED CLASS LEGEND (both sensors, final) ----------------
MASK_CLEAR, MASK_FRESH, MASK_OLD = 0, 2, 3
MASK_CLOUD, MASK_SHADOW, MASK_WATER, MASK_NODATA = 4, 5, 6, 255
FRESH_MAX_DAYS = 90
MASK_COLORMAP = {0:(200,200,200,255), 2:(220,40,40,255), 3:(140,80,40,255),
                 4:(255,255,255,255), 5:(60,60,90,255), 6:(30,90,200,255), 255:(0,0,0,0)}

DESTRUCTIVE = False   # False: write *_mask_v2.tif alongside; True: overwrite *_mask.tif

# ---------------- paths & radiometry ----------------
EFFIS_PATH = "data/PHISAT_5fbf994ef5774ab882bf7239ec8b32f5.zip"
PHISAT_GLOB = "output/acquisitions/*/*"
HEO_DIR = "data/HEO_dataset"
TILE, OUT_ROOT = 256, "processed/dataset_v1"
QUANTIFICATION = {"heo": 4094.0, "phisat2_sim": 10000.0}

HEO_BAND_MAP = dict(pan=1, blue=2, green=3, red=4, re1=5, re2=6, re3=7, nir=8)  # 1-based, B00..B07
HEO_TRAIN_BANDS = [2, 3, 4, 5, 6, 7, 8]   # drop PAN -> identical 7-ch layout to PhiSat-2
BAND_NAMES = ["B", "G", "R", "RE1", "RE2", "RE3", "NIR"]

# EFFIS back to 2020, all countries, NO area floor (small old scars must label too)
import sys; sys.path.insert(0, ".")
from importlib import reload
# assumes load_effis + helpers are importable; else paste the loader cell here
exec(open("effis_loader.py").read()) if os.path.exists("effis_loader.py") else None
fires_all = load_effis(EFFIS_PATH, countries=["IT","FR","ES","EL"],
                       start_date="2020-01-01", end_date="2026-08-04", min_area_ha=0)
log.info("%d EFFIS polygons loaded", len(fires_all))

2026-08-05 13:03:46,752 INFO 25707 EFFIS polygons loaded


In [2]:
def rasterize_sel(sel, shape, transform):
    if len(sel) == 0: return np.zeros(shape, bool)
    return rasterize(((g, 1) for g in sel.geometry), out_shape=shape,
                     transform=transform, fill=0, dtype=np.uint8).astype(bool)

# old legend -> new legend LUT (burn split handled separately)
LUT = np.zeros(256, np.uint8)
LUT[0], LUT[3], LUT[4], LUT[5], LUT[255] = MASK_CLEAR, MASK_CLOUD, MASK_SHADOW, MASK_WATER, MASK_NODATA
OLD_LEGEND_BURN = 2

relabeled = []
for old_path in sorted(glob.glob(f"{PHISAT_GLOB}/*_mask.tif")):
    if old_path.endswith("_mask_v2.tif"): continue
    out_path = old_path if DESTRUCTIVE else old_path.replace("_mask.tif", "_mask_v2.tif")
    if not DESTRUCTIVE and os.path.exists(out_path): continue   # resume-safe

    ts_str = os.path.basename(os.path.dirname(old_path))
    acq = np.datetime64(dt.datetime.strptime(ts_str, "%Y-%m-%dT%H-%M-%S"))
    with rasterio.open(old_path) as src:
        old = src.read(1); crs, transform, prof = src.crs, src.transform, src.profile

    fires_utm = fires_all.to_crs(crs)
    age = (acq - fires_utm["fire_date"].values).astype("timedelta64[D]").astype(int)
    fresh_r = rasterize_sel(fires_utm[(age >= 0) & (age <= FRESH_MAX_DAYS)], old.shape, transform)
    old_r   = rasterize_sel(fires_utm[age > FRESH_MAX_DAYS], old.shape, transform)

    new = LUT[old]
    was_burn = old == OLD_LEGEND_BURN
    new[was_burn & fresh_r] = MASK_FRESH
    new[was_burn & ~fresh_r] = MASK_OLD
    # scars pre-dating the original 365d cap: add as old burn, without stealing
    # pixels that cloud/shadow won on priority
    new[old_r & np.isin(new, [MASK_CLEAR, MASK_WATER])] = MASK_OLD

    prof.update(nodata=MASK_NODATA)
    with rasterio.open(out_path, "w", **prof) as dst:
        dst.write_colormap(1, MASK_COLORMAP)
        dst.write(new, 1)
    counts = {int(k): int(v) for k, v in zip(*np.unique(new, return_counts=True))}
    relabeled.append(dict(mask=os.path.basename(out_path), **{f"c{k}": v for k, v in counts.items()}))
    log.info("%s %s", os.path.basename(out_path), counts)

pd.DataFrame(relabeled).to_csv("output/relabel_v2_index.csv", index=False)
print(f"relabeled {len(relabeled)} masks")

relabeled 0 masks


In [5]:
def parse_heo_datetime(name):
    for tok in name.split("_"):
        if len(tok) == 15 and tok[8] == "T":
            return dt.datetime.strptime(tok, "%Y%m%dT%H%M%S")
    raise ValueError(name)

def load_heo_stack(product_dir, ref_band="B01"):
    """B00-B07 single-band files -> (8,H,W) on the exact 2.5 m grid of ref_band.
    5 m bands (RE2/RE3) are reprojected, not naively resized (5515 != 2*2757)."""
    base = os.path.basename(product_dir)
    paths = {i: os.path.join(product_dir, "IMG_DATA", f"{base}.B{i:02d}.tif") for i in range(8)}
    with rasterio.open(paths[int(ref_band[1:])]) as ref:
        H, W, crs, transform = ref.height, ref.width, ref.crs, ref.transform
    stack = np.zeros((8, H, W), np.uint16)
    for i in range(8):
        with rasterio.open(paths[i]) as src:
            if (src.height, src.width) == (H, W):
                stack[i] = src.read(1)
            else:
                reproject(rasterio.band(src, 1), stack[i],
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=transform, dst_crs=crs,
                          resampling=Resampling.bilinear)
    return stack, crs, transform

def build_heo_mask(product_dir, device="cuda:0"):
    from omnicloudmask import predict_from_array
    import cv2
    acq = parse_heo_datetime(os.path.basename(product_dir))
    stack, crs, transform = load_heo_stack(product_dir)
    valid = ~np.all(stack == 0, axis=0)

    # OCM at ~10 m (its training GSD), mask upsampled back to 2.5 m
    r, g, nir = (stack[HEO_BAND_MAP[k]-1].astype(np.float32) for k in ("red","green","nir"))
    f = 4  # 2.5 m -> 10 m
    small = np.stack([cv2.resize(b, (r.shape[1]//f, r.shape[0]//f),
                                 interpolation=cv2.INTER_AREA) for b in (r, g, nir)])
    pred = np.asarray(predict_from_array(small, batch_size=1, apply_no_data_mask=True,
                                         no_data_value=0, inference_device=device))[0]
    pred = cv2.resize(pred.astype(np.uint8), r.shape[::-1], interpolation=cv2.INTER_NEAREST)
    cloud, shadow = ((pred == 1) | (pred == 2)) & valid, (pred == 3) & valid

    ndwi = (g - nir) / np.maximum(g + nir, 1e-6)
    water = (ndwi > 0.05) & valid

    fires_utm = fires_all.to_crs(crs)
    age = (np.datetime64(acq) - fires_utm["fire_date"].values).astype("timedelta64[D]").astype(int)
    fresh_r = rasterize_sel(fires_utm[(age >= 0) & (age <= FRESH_MAX_DAYS)], r.shape, transform)
    old_r   = rasterize_sel(fires_utm[age > FRESH_MAX_DAYS], r.shape, transform)

    lab = np.full(r.shape, MASK_CLEAR, np.uint8)   # priority identical to PhiSat-2
    lab[water] = MASK_WATER
    lab[old_r] = MASK_OLD
    lab[fresh_r] = MASK_FRESH
    lab[shadow] = MASK_SHADOW
    lab[cloud] = MASK_CLOUD
    lab[~valid] = MASK_NODATA

    out = os.path.join(product_dir, f"{os.path.basename(product_dir)}_mask.tif")
    prof = dict(driver="GTiff", height=lab.shape[0], width=lab.shape[1], count=1,
                dtype="uint8", crs=crs, transform=transform, nodata=MASK_NODATA,
                compress="deflate", tiled=True, blockxsize=512, blockysize=512)
    with rasterio.open(out, "w", **prof) as dst:
        dst.write_colormap(1, MASK_COLORMAP)
        dst.write(lab, 1)
    counts = {int(k): int(v) for k, v in zip(*np.unique(lab, return_counts=True))}
    log.info("%s (acq %s): %s", os.path.basename(product_dir), acq.date(), counts)
    return counts

heo_rows = []
for prod in sorted(d for d in glob.glob(f"{HEO_DIR}/IMH0*") if os.path.isdir(d)):
    try:
        heo_rows.append(dict(product=os.path.basename(prod),
                             **{f"c{k}": v for k, v in build_heo_mask(prod).items()}))
    except Exception:
        log.exception("%s failed", prod)
pd.DataFrame(heo_rows).to_csv(f"{HEO_DIR}/heo_mask_index.csv", index=False)

2026-08-05 13:07:28,715 INFO IMH01_1CST__OPT8_20250709T100945_20250709T100947_20260330T172935_02623______O_A02 (acq 2025-07-09): {0: 11285231, 2: 56318, 3: 1285103, 4: 4421074, 5: 2390582, 6: 2005902, 255: 8921380}
2026-08-05 13:07:38,334 INFO IMH02_1CST__OPT8_20251115T104743_20251115T104745_20260511T102716_02159______O_A02 (acq 2025-11-15): {0: 15057006, 4: 3487090, 5: 2663807, 6: 26, 255: 8314573}
2026-08-05 13:07:48,853 INFO IMH02_1CST__OPT8_20260711T125448_20260711T125450_20260712T174910_05716______O_A02 (acq 2026-07-11): {0: 21258423, 3: 206167, 4: 26992, 5: 22384, 6: 1498, 255: 8838820}
2026-08-05 13:07:59,528 INFO IMH02_1CST__OPT8_20260711T125501_20260711T125504_20260712T175013_05716______O_A02 (acq 2026-07-11): {0: 9930255, 3: 12108, 4: 11309401, 5: 236348, 6: 22118, 255: 8986784}
2026-08-05 13:08:09,756 INFO IMH05_1CST__OPT8_20260622T122713_20260622T122715_20260624T163803_05433______O_A02 (acq 2026-06-22): {0: 21602933, 3: 162859, 6: 1815, 255: 8752383}
2026-08-05 13:08:21,177

In [4]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# legend -> colors (matches the GeoTIFF colormap)
CLS = [(0,"clear",(.78,.78,.78)), (2,"fresh burn",(.86,.16,.16)), (3,"old burn",(.55,.31,.16)),
       (4,"cloud",(1,1,1)), (5,"shadow",(.24,.24,.35)), (6,"water",(.12,.35,.78)),
       (255,"nodata",(0,0,0))]
vals = [c[0] for c in CLS]
cmap = ListedColormap([c[2] for c in CLS])
norm = BoundaryNorm([v-0.5 for v in vals] + [255.5], cmap.N)
legend_patches = [Patch(color=c[2], label=f"{c[0]} {c[1]}") for c in CLS]

def stretch_rgb(img, bands_rgb, scale, p=(2, 98)):
    """img: (C,H,W) raw DN -> (H,W,3) display-stretched reflectance."""
    rgb = img[list(bands_rgb)].astype(np.float32) / scale
    out = np.zeros((*rgb.shape[1:], 3), np.float32)
    for i in range(3):
        lo, hi = np.percentile(rgb[i][rgb[i] > 0], p) if (rgb[i] > 0).any() else (0, 1)
        out[..., i] = np.clip((rgb[i] - lo) / max(hi - lo, 1e-6), 0, 1)
    return out

def qc_panel(img, mask, title, bands_rgb, scale, downsample=4):
    im, mk = img[:, ::downsample, ::downsample], mask[::downsample, ::downsample]
    rgb = stretch_rgb(im, bands_rgb, scale)
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    ax[0].imshow(rgb); ax[0].set_title(f"{title}\nRGB")
    ax[1].imshow(mk, cmap=cmap, norm=norm, interpolation="nearest"); ax[1].set_title("mask")
    ax[2].imshow(rgb); ax[2].imshow(np.ma.masked_where(mk == 0, mk),
                                    cmap=cmap, norm=norm, alpha=0.45,
                                    interpolation="nearest"); ax[2].set_title("overlay")
    for a in ax: a.axis("off")
    fig.legend(handles=legend_patches, loc="lower center", ncol=7, frameon=False)
    plt.tight_layout(); plt.show()

mask_suffix = "_mask.tif" if DESTRUCTIVE else "_mask_v2.tif"

# ---- PhiSat-2: show the N scenes with the most fresh burn (most informative QC) ----
ph = pd.read_csv("output/relabel_v2_index.csv")
ph["fresh_px"] = ph.get("c2", 0)
for _, row in ph.sort_values("fresh_px", ascending=False).head(30).iterrows():
    mp = glob.glob(f"{PHISAT_GLOB}/{row['mask']}")[0]
    ip = mp.replace(mask_suffix, "_phisat2.tif")
    with rasterio.open(ip) as s: img = s.read()
    with rasterio.open(mp) as s: msk = s.read(1)
    # exported band order B,G,R,... -> RGB display = (R,G,B) = indices (2,1,0)
    qc_panel(img, msk, os.path.basename(ip), bands_rgb=(2, 1, 0), scale=10000.0)

# ---- HEO: all 9 products ----
for prod in sorted(d for d in glob.glob(f"{HEO_DIR}/IMH0*") if os.path.isdir(d)):
    mp = os.path.join(prod, f"{os.path.basename(prod)}_mask.tif")
    if not os.path.exists(mp): continue
    stack, _, _ = load_heo_stack(prod)
    img7 = stack[np.array(HEO_TRAIN_BANDS) - 1]     # same 7-ch order as PhiSat-2
    with rasterio.open(mp) as s: msk = s.read(1)
    qc_panel(img7, msk, os.path.basename(prod)[:45], bands_rgb=(2, 1, 0), scale=4094.0)

EmptyDataError: No columns to parse from file